# Modelo Diputrax — Predicción de Acceso por Mayoría Relativa

Clasificación binaria: ¿llega un diputado federal vía **mayoría relativa (MR)** o vía **representación proporcional (RP)**?  
Enfoque análogo al modelo FairLend: interpretable, auditado, con diagnósticos de equidad.

---
## 1. Contexto y planteamiento del problema

**Contexto.** La Cámara de Diputados de México combina dos vías de acceso: los 300 escaños de MR (el candidato con más votos en cada distrito gana) y los 200 escaños de RP (listas de partido). Las características socioeconómicas y de trayectoria de quienes acceden por cada vía revelan patrones de selección y equidad de representación.

**Problema.** ¿Pueden las características demográficas, de formación y de trayectoria política previa predecir la vía de acceso de un diputado? Si el modelo aprende sesgos sistemáticos (género, escolaridad, partido), esas diferencias cuantifican barreras estructurales de representación.

**Implicación.** El modelo debe balancear desempeño predictivo con interpretabilidad, generar códigos de razón auditables y exponer atributos asociados a la vía de acceso.

---
## 2. Objetivo y criterios de éxito

**Objetivo primario.** Construir un modelo de clasificación interpretable que prediga `mayoria_relativa` y exponga los predictores más influyentes para diagnóstico de equidad.

**Objetivos secundarios.**
- Identificar variables asociadas a la vía MR.
- Cuantificar equidad por partido, género y legislatura.
- Generar códigos de razón (SHAP) para decisiones individuales.

**Métricas clave.**
- ROC-AUC y PR-AUC (desempeño general).
- Recall clase 1 (MR): minimizar falsos negativos si el objetivo es detectar acceso por MR.
- F1-score macro (balance entre clases).

---
## 3. Diccionario de datos (variables seleccionadas)

| Variable | Tipo | Descripción |
|---|---|---|
| `mayoria_relativa` | **Target (0/1)** | 1 = ganó su distrito (MR); 0 = entró por lista (RP) |
| `edad_al_tomar_cargo` | Numérico | Edad en años al asumir el cargo |
| `grado_estudios_ord` | Numérico ordinal | Nivel educativo (0=sin dato … 7=doctorado) |
| `n_cargos_legislativos_prev` | Numérico | Número de cargos legislativos previos |
| `n_comisiones` | Numérico | Comisiones ordinarias en las que participó |
| `n_comisiones_especiales` | Numérico | Comisiones especiales |
| `n_presidencias` | Numérico | Veces que presidió una comisión |
| `n_secretarias` | Numérico | Veces que fue secretario de comisión |
| `n_trayectoria_admin` | Numérico | Cargos en administración pública |
| `n_trayectoria_legislativa` | Numérico | Cargos legislativos en trayectoria total |
| `n_trayectoria_politica` | Numérico | Cargos político-partidistas |
| `n_trayectoria_empresarial` | Numérico | Cargos en sector privado/empresarial |
| `fue_diputado_local` | Binario | Fue diputado local antes |
| `fue_diputado_federal` | Binario | Fue diputado federal antes |
| `fue_senador` | Binario | Fue senador antes |
| `legislatura_num` | Numérico | Número de legislatura |
| `partido` | Categórico | Partido político |
| `area_formacion` | Categórico | Área de formación académica |

---
## 4. Configuración e imports

In [ ]:
import os
import glob
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Preprocessing
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.impute import SimpleImputer
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from xgboost import XGBClassifier

# Metrics
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, auc,
    recall_score, precision_score, accuracy_score, f1_score
)
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Interpretability
import shap

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
RANDOM_STATE = 42

---
## 5. Carga de datos — snapshot ETL más reciente

In [ ]:
ETL_ROOT = Path('/home/miso/Projects/legisdatamxsil/data/etl')

snapshots = sorted([d for d in ETL_ROOT.iterdir() if d.is_dir()])
latest = snapshots[-1]
print(f'Snapshots : {[d.name for d in snapshots]}')
print(f'Using     : {latest.name}')

csv_files = sorted(latest.glob('*.csv'))
frames = [pd.read_csv(f, low_memory=False) for f in csv_files]
df_raw = pd.concat(frames, ignore_index=True)
print(f'\nRaw shape : {df_raw.shape}')
df_raw.head(3)

---
## 6. Selección de features y target

In [ ]:
TARGET = 'mayoria_relativa'

NUMERIC_FEATURES = [
    'edad_al_tomar_cargo',
    'grado_estudios_ord',
    'n_cargos_legislativos_prev',
    'n_comisiones',
    'n_comisiones_especiales',
    'n_presidencias',
    'n_secretarias',
    'n_trayectoria_admin',
    'n_trayectoria_legislativa',
    'n_trayectoria_politica',
    'n_trayectoria_empresarial',
    'fue_diputado_local',
    'fue_diputado_federal',
    'fue_senador',
    'legislatura_num',
]

CATEGORICAL_FEATURES = ['partido', 'area_formacion']

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

# Keep only columns that exist in the dataset
available_num = [c for c in NUMERIC_FEATURES if c in df_raw.columns]
available_cat = [c for c in CATEGORICAL_FEATURES if c in df_raw.columns]
available_all = available_num + available_cat

missing_cols = set(ALL_FEATURES) - set(available_all)
if missing_cols:
    print(f'WARNING — columns not found, will be skipped: {missing_cols}')

NUMERIC_FEATURES = available_num
CATEGORICAL_FEATURES = available_cat
ALL_FEATURES = available_all

print(f'Target           : {TARGET}')
print(f'Numeric features : {len(NUMERIC_FEATURES)}')
print(f'Categorical feats: {len(CATEGORICAL_FEATURES)}')

df = df_raw[[TARGET] + ALL_FEATURES].copy()
print(f'Working shape    : {df.shape}')

---
## 7. Calidad de datos y valores faltantes

In [ ]:
print('=== Target completeness ===')
print(df[TARGET].value_counts(dropna=False))
print(f'\nClass balance: {df[TARGET].mean():.2%} MR (1)')

# Missing value audit
miss = df.isnull().mean().sort_values(ascending=False)
miss_nonzero = miss[miss > 0]

print(f'\n=== Missing values (non-zero) ===')
print(miss_nonzero.to_string())

# Rows with all predictors missing
rows_no_data = df[ALL_FEATURES].isna().all(axis=1)
print(f'\nRows with ALL predictors missing: {rows_no_data.sum()}')
print(f'Rows with ANY predictor missing : {df[ALL_FEATURES].isna().any(axis=1).sum()}')
print(f'Complete predictor rows         : {df[ALL_FEATURES].notna().all(axis=1).sum()}')

In [ ]:
if miss_nonzero.shape[0] > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(miss_nonzero) * 0.35)))
    miss_nonzero.plot.barh(ax=ax, color='steelblue')
    ax.set_xlabel('Fracción faltante')
    ax.set_title('Valores faltantes por variable')
    plt.tight_layout()
    plt.show()

---
## 8. Análisis exploratorio orientado al modelo

### 8.1 Distribución del target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df[TARGET].value_counts()
axes[0].bar(['RP (0)', 'MR (1)'], counts.values, color=['#4c72b0', '#dd8452'], edgecolor='white')
axes[0].set_title('Distribución: Mayoría Relativa vs RP')
axes[0].set_ylabel('Diputados')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 20, f'{v}\n({v/len(df):.1%})', ha='center', fontsize=10)

if 'legislatura_num' in df.columns:
    pivot = df.groupby('legislatura_num')[TARGET].mean()
    pivot.plot(ax=axes[1], marker='o', color='steelblue')
    axes[1].axhline(df[TARGET].mean(), ls='--', color='red', label='media global')
    axes[1].set_title('% MR por legislatura')
    axes[1].set_ylabel('Proporción MR')
    axes[1].legend()

plt.tight_layout()
plt.show()

### 8.2 Features numéricas vs target

In [ ]:
key_num = [c for c in ['edad_al_tomar_cargo', 'grado_estudios_ord',
                        'n_cargos_legislativos_prev', 'n_trayectoria_legislativa',
                        'n_trayectoria_politica', 'n_trayectoria_admin'] if c in df.columns]

n = len(key_num)
fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_num):
    for label, grp in df.groupby(TARGET):
        grp[col].dropna().plot.hist(
            ax=axes[i], alpha=0.5, bins=25,
            label='MR' if label == 1 else 'RP',
            color='#dd8452' if label == 1 else '#4c72b0'
        )
    axes[i].set_title(col)
    axes[i].legend()

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de features numéricas por vía de acceso', y=1.02)
plt.tight_layout()
plt.show()

### 8.3 Features categóricas vs target

In [ ]:
for cat_col in CATEGORICAL_FEATURES:
    if cat_col not in df.columns:
        continue
    top_cats = df[cat_col].value_counts().head(12).index
    sub = df[df[cat_col].isin(top_cats)]
    mr_rate = sub.groupby(cat_col)[TARGET].mean().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(12, 4))
    mr_rate.plot.bar(ax=ax, color='teal', edgecolor='white')
    ax.axhline(df[TARGET].mean(), ls='--', color='red', label='media global')
    ax.set_ylabel('Proporción MR')
    ax.set_title(f'Tasa MR por {cat_col} (top 12)')
    ax.legend()
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### 8.4 Correlación con el target

In [ ]:
num_target_corr = df[NUMERIC_FEATURES + [TARGET]].corr()[TARGET].drop(TARGET).sort_values()

fig, ax = plt.subplots(figsize=(8, max(4, len(num_target_corr) * 0.35)))
colors = ['#dd8452' if v > 0 else '#4c72b0' for v in num_target_corr.values]
num_target_corr.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f'Correlación de Pearson con {TARGET}')
plt.tight_layout()
plt.show()

---
## 9. Preprocesamiento

### 9.1 Limpieza inicial

In [ ]:
# Drop rows where target is missing
df = df.dropna(subset=[TARGET]).copy()
print(f'After dropping missing target: {df.shape}')

# Drop rows with ALL predictors missing
rows_no_data = df[ALL_FEATURES].isna().all(axis=1)
df = df[~rows_no_data].copy()
print(f'After dropping all-missing rows: {df.shape}')

df[TARGET] = df[TARGET].astype(int)

### 9.2 Indicadores de valores faltantes

Columnas faltantes pueden ser informativamente no-aleatorias (MNAR). Añadimos indicadores binarios.

In [ ]:
missing_indicator_cols = []

for col in NUMERIC_FEATURES + CATEGORICAL_FEATURES:
    if df[col].isna().sum() > 0:
        ind_col = f'MISSING_{col.upper()}'
        df[ind_col] = df[col].isna().astype(int)
        missing_indicator_cols.append(ind_col)

print(f'Missing indicator columns created: {missing_indicator_cols}')

### 9.3 Train/test split (stratificado)

In [ ]:
feature_cols = ALL_FEATURES + missing_indicator_cols

X = df[feature_cols].copy()
y = df[TARGET].copy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f'Train: {X_train_raw.shape} | default rate: {y_train.mean():.3f}')
print(f'Test : {X_test_raw.shape} | default rate: {y_test.mean():.3f}')

### 9.4 Pipelines de imputación y codificación

Se construyen dos versiones: **Simple** (mediana/moda) y **MICE** (imputación iterativa multivariada).

In [ ]:
# Separate indicator columns (already binary, no imputation needed)
num_cols_model = [c for c in NUMERIC_FEATURES if c in X.columns]
cat_cols_model = [c for c in CATEGORICAL_FEATURES if c in X.columns]
ind_cols_model = missing_indicator_cols

def build_preprocessor(imputation='simple'):
    """Return a ColumnTransformer for numeric+categorical+indicator columns."""
    if imputation == 'simple':
        num_imputer = SimpleImputer(strategy='median')
    else:  # mice
        num_imputer = IterativeImputer(max_iter=10, random_state=RANDOM_STATE)

    numeric_pipe = Pipeline([
        ('imputer', num_imputer),
        ('scaler', StandardScaler())
    ])

    categorical_pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])

    steps = []
    if num_cols_model:
        steps.append(('num', numeric_pipe, num_cols_model))
    if cat_cols_model:
        steps.append(('cat', categorical_pipe, cat_cols_model))
    if ind_cols_model:
        # Indicators: pass-through (already 0/1)
        steps.append(('ind', 'passthrough', ind_cols_model))

    return ColumnTransformer(steps, remainder='drop')


prep_simple = build_preprocessor('simple')
prep_mice   = build_preprocessor('mice')

X_train_simple = prep_simple.fit_transform(X_train_raw)
X_test_simple  = prep_simple.transform(X_test_raw)

X_train_mice   = prep_mice.fit_transform(X_train_raw)
X_test_mice    = prep_mice.transform(X_test_raw)


def get_feature_names(prep):
    names = []
    for name, trans, cols in prep.transformers_:
        if name == 'num':
            names += cols
        elif name == 'cat':
            enc = trans.named_steps['encoder']
            names += list(enc.get_feature_names_out(cols))
        elif name == 'ind':
            names += cols
    return names


feat_names_simple = get_feature_names(prep_simple)
feat_names_mice   = get_feature_names(prep_mice)

print(f'Feature count (Simple): {len(feat_names_simple)}')
print(f'Feature count (MICE)  : {len(feat_names_mice)}')

### 9.5 VIF — Diagnóstico de multicolinealidad (features numéricas)

In [ ]:
num_idx = [i for i, n in enumerate(feat_names_simple) if n in num_cols_model]
X_num_vif = X_train_simple[:, num_idx]

vif_data = pd.DataFrame({
    'feature': [feat_names_simple[i] for i in num_idx],
    'VIF': [variance_inflation_factor(X_num_vif, i) for i in range(X_num_vif.shape[1])]
}).sort_values('VIF', ascending=False)

print(vif_data.to_string(index=False))
high_vif = vif_data[vif_data['VIF'] > 10]
if not high_vif.empty:
    print(f'\n⚠ Features with VIF > 10 (potential multicollinearity): {high_vif["feature"].tolist()}')

---
## 10. Entrenamiento de modelos

Se entrena sobre imputación **Simple** como línea base y **MICE** para comparar.  
Modelos: Logistic Regression, Decision Tree, Random Forest, XGBoost.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def evaluate(name, model, X_tr, X_te, y_tr, y_te, feat_names=None):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    cv_scores = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1)

    result = {
        'model': name,
        'accuracy': accuracy_score(y_te, y_pred),
        'recall_MR': recall_score(y_te, y_pred),
        'precision_MR': precision_score(y_te, y_pred),
        'f1_macro': f1_score(y_te, y_pred, average='macro'),
        'roc_auc': roc_auc_score(y_te, y_prob),
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std': cv_scores.std(),
        '_model_obj': model,
        '_y_prob': y_prob,
    }
    return result


MODELS = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, class_weight='balanced', random_state=RANDOM_STATE
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=5,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
        eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
    ),
}

results_simple = []
results_mice   = []

for mname, mobj in MODELS.items():
    import copy
    print(f'Training {mname} ...')
    r_s = evaluate(mname, copy.deepcopy(mobj), X_train_simple, X_test_simple, y_train, y_test, feat_names_simple)
    r_m = evaluate(mname, copy.deepcopy(mobj), X_train_mice,   X_test_mice,   y_train, y_test, feat_names_mice)
    results_simple.append(r_s)
    results_mice.append(r_m)

print('Done.')

---
## 11. Comparación de modelos

In [ ]:
def results_df(results, label):
    cols = ['model', 'accuracy', 'recall_MR', 'precision_MR', 'f1_macro', 'roc_auc', 'cv_auc_mean', 'cv_auc_std']
    df_r = pd.DataFrame(results)[cols].set_index('model')
    df_r.index = [f'{n} ({label})' for n in df_r.index]
    return df_r

summary = pd.concat([
    results_df(results_simple, 'Simple'),
    results_df(results_mice,   'MICE')
]).sort_values('roc_auc', ascending=False)

summary.style.background_gradient(subset=['roc_auc', 'recall_MR', 'f1_macro'], cmap='YlGn')

In [ ]:
# ROC curves — Simple imputation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for r in results_simple:
    fpr, tpr, _ = roc_curve(y_test, r['_y_prob'])
    axes[0].plot(fpr, tpr, label=f"{r['model']} (AUC={r['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[0].set_title('ROC — Imputación Simple')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(fontsize=8)

for r in results_mice:
    fpr, tpr, _ = roc_curve(y_test, r['_y_prob'])
    axes[1].plot(fpr, tpr, label=f"{r['model']} (AUC={r['roc_auc']:.3f})")
axes[1].plot([0, 1], [0, 1], 'k--', lw=0.8)
axes[1].set_title('ROC — MICE')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrices — best model per imputation
best_simple = max(results_simple, key=lambda r: r['roc_auc'])
best_mice   = max(results_mice,   key=lambda r: r['roc_auc'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, r, label in [
    (axes[0], best_simple, 'Simple'),
    (axes[1], best_mice,   'MICE')
]:
    y_pred = r['_model_obj'].predict(X_test_simple if label == 'Simple' else X_test_mice)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['RP (0)', 'MR (1)'],
                yticklabels=['RP (0)', 'MR (1)'])
    ax.set_title(f"{r['model']} — {label}\nAUC={r['roc_auc']:.3f}")
    ax.set_ylabel('Real'); ax.set_xlabel('Predicho')

plt.tight_layout()
plt.show()

print('\n=== Best Simple ===')
y_pred_s = best_simple['_model_obj'].predict(X_test_simple)
print(classification_report(y_test, y_pred_s, target_names=['RP', 'MR']))

print('\n=== Best MICE ===')
y_pred_m = best_mice['_model_obj'].predict(X_test_mice)
print(classification_report(y_test, y_pred_m, target_names=['RP', 'MR']))

---
## 12. Ajuste fino del Random Forest (GridSearchCV)

Análogo a FairLend: el RF con class_weight ajustado maximiza recall en la clase de interés.

In [ ]:
param_grid_rf = {
    'n_estimators': [200, 400],
    'max_depth': [None, 10, 20],
    'min_samples_leaf': [1, 5],
    'class_weight': ['balanced', {0: 1, 1: 3}],
}

rf_base = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

gs_rf = GridSearchCV(
    rf_base, param_grid_rf,
    scoring='recall',
    cv=cv, n_jobs=-1, verbose=1
)
gs_rf.fit(X_train_simple, y_train)

print(f'Best params  : {gs_rf.best_params_}')
print(f'Best CV recall: {gs_rf.best_score_:.4f}')

In [ ]:
rf_tuned = gs_rf.best_estimator_
y_pred_rf = rf_tuned.predict(X_test_simple)
y_prob_rf = rf_tuned.predict_proba(X_test_simple)[:, 1]

print('=== Random Forest (fine-tuned) — Test ===')
print(classification_report(y_test, y_pred_rf, target_names=['RP', 'MR']))
print(f'ROC-AUC: {roc_auc_score(y_test, y_prob_rf):.4f}')

---
## 13. Importancia de features

### 13.1 Logistic Regression — coeficientes

In [ ]:
lr_model = next(r['_model_obj'] for r in results_simple if r['model'] == 'Logistic Regression')

coef_df = pd.DataFrame({
    'feature': feat_names_simple,
    'coef': lr_model.coef_[0],
    'abs_coef': np.abs(lr_model.coef_[0])
}).sort_values('abs_coef', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#dd8452' if c > 0 else '#4c72b0' for c in coef_df['coef']]
ax.barh(coef_df['feature'], coef_df['coef'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression — Top 20 coeficientes (magnitude)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 13.2 Random Forest — feature importance (Gini)

In [ ]:
fi_df = pd.DataFrame({
    'feature': feat_names_simple,
    'importance': rf_tuned.feature_importances_
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
fi_df.set_index('feature')['importance'].plot.barh(ax=ax, color='teal', edgecolor='white')
ax.set_title('Random Forest — Top 20 features (Gini importance)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

---
## 14. SHAP — Análisis de interpretabilidad

### 14.1 SHAP global — Logistic Regression

In [ ]:
X_test_simple_df = pd.DataFrame(X_test_simple, columns=feat_names_simple)
X_train_simple_df = pd.DataFrame(X_train_simple, columns=feat_names_simple)

explainer_lr = shap.LinearExplainer(lr_model, X_train_simple_df)
shap_values_lr = explainer_lr(X_test_simple_df)

shap.summary_plot(shap_values_lr, X_test_simple_df, show=True, max_display=20)

### 14.2 SHAP global — Random Forest (fine-tuned)

In [ ]:
# Use a subsample to keep runtime reasonable
sample_size = min(500, len(X_test_simple_df))
X_shap_sample = X_test_simple_df.sample(sample_size, random_state=RANDOM_STATE)

explainer_rf = shap.TreeExplainer(rf_tuned)
shap_values_rf = explainer_rf(X_shap_sample)

# For binary classification, TreeExplainer returns shape (n, features, 2) — take class 1
sv = shap_values_rf.values
if sv.ndim == 3:
    sv = sv[:, :, 1]

shap.summary_plot(sv, X_shap_sample, show=True, max_display=20)

### 14.3 SHAP local — códigos de razón (waterfall plot)

In [ ]:
# Show reason codes for one MR prediction and one RP prediction
y_pred_sample = rf_tuned.predict(X_shap_sample.values)
mr_idx  = np.where(y_pred_sample == 1)[0][0]
rp_idx  = np.where(y_pred_sample == 0)[0][0]

for idx, label in [(mr_idx, 'MR (predicho)'), (rp_idx, 'RP (predicho)')]:
    print(f'\n--- Diputado #{idx}: {label} ---')
    explanation = shap.Explanation(
        values=sv[idx],
        base_values=explainer_rf.expected_value[1] if hasattr(explainer_rf.expected_value, '__len__') else explainer_rf.expected_value,
        data=X_shap_sample.iloc[idx].values,
        feature_names=feat_names_simple
    )
    shap.plots.waterfall(explanation, max_display=15)

---
## 15. Diagnósticos de equidad

¿El modelo reproduce sesgos por partido o por legislatura?

In [ ]:
# Reattach metadata to test set
test_idx = y_test.index
df_test_meta = df.loc[test_idx].copy()
df_test_meta['y_true'] = y_test.values
df_test_meta['y_pred'] = rf_tuned.predict(X_test_simple)
df_test_meta['y_prob'] = rf_tuned.predict_proba(X_test_simple)[:, 1]

fairness_cols = [c for c in ['partido', 'legislatura_num'] if c in df_test_meta.columns]

for fcol in fairness_cols:
    if fcol == 'partido':
        top_groups = df_test_meta['partido'].value_counts().head(10).index
        sub = df_test_meta[df_test_meta['partido'].isin(top_groups)]
    else:
        sub = df_test_meta

    grp = sub.groupby(fcol).apply(
        lambda g: pd.Series({
            'n': len(g),
            'true_MR_rate': g['y_true'].mean(),
            'pred_MR_rate': g['y_pred'].mean(),
            'recall': recall_score(g['y_true'], g['y_pred'], zero_division=0),
            'precision': precision_score(g['y_true'], g['y_pred'], zero_division=0),
        })
    ).sort_values('true_MR_rate', ascending=False)

    print(f'\n=== Equidad por {fcol} ===')
    print(grp.to_string())

    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(grp))
    ax.bar(x - 0.2, grp['true_MR_rate'], 0.4, label='Tasa MR real', color='#4c72b0')
    ax.bar(x + 0.2, grp['pred_MR_rate'], 0.4, label='Tasa MR predicha', color='#dd8452')
    ax.set_xticks(x)
    ax.set_xticklabels(grp.index, rotation=45, ha='right')
    ax.set_title(f'Equidad — tasa MR real vs predicha por {fcol}')
    ax.legend()
    plt.tight_layout()
    plt.show()

---
## 16. Resumen ejecutivo

In [ ]:
print('=' * 60)
print('RESUMEN EJECUTIVO — Modelo Diputrax')
print('=' * 60)

print(f"""
Objetivo    : predecir vía de acceso (MR=1 vs RP=0).
Registros   : {len(df):,} diputados en dataset de trabajo.
Split       : 80% train / 20% test (estratificado).
Imputación  : Simple (mediana/moda) y MICE comparados.
Indicadores : {len(missing_indicator_cols)} columnas de missingness añadidas.
""")

print('--- Scoreboard (test) ---')
print(summary[['accuracy', 'recall_MR', 'precision_MR', 'f1_macro', 'roc_auc']].to_string())

rf_auc   = roc_auc_score(y_test, y_prob_rf)
rf_rec   = recall_score(y_test, y_pred_rf)
rf_prec  = precision_score(y_test, y_pred_rf)
rf_acc   = accuracy_score(y_test, y_pred_rf)

print(f"""
--- Champion: Random Forest (fine-tuned) ---
ROC-AUC    : {rf_auc:.4f}
Recall MR  : {rf_rec:.4f}
Precision MR: {rf_prec:.4f}
Accuracy   : {rf_acc:.4f}

Recomendación:
  - Random Forest fine-tuned como modelo de screening.
  - Logistic Regression como challenger auditable.
  - XGBoost como benchmark de precisión.
  - Revisar partidos/legislaturas con mayor divergencia
    real vs predicho en diagnóstico de equidad.
""")